## Library Imports

In [1]:
!pip install fastavro pandas
!pip install faker
!pip install timezonefinder

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 MB 14.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.8 MB/s eta 0:00:00
  Created wheel for timezonefinder: filename=timezonefinder-6.5.0-cp310-cp310-manylinux_2_35_x86_64.whl size=49412387 sha256=46b750ac3525e87d916a2cda4155c87e080fcfcdd281b207d134385ca09d499b
  Stored in directory: /root/.cache/pip/wheels/f2/ae/2d/8bb82cff928de1ce64feef734cf9ef78d84f026693fecc5617
Successfully built timezonefinder


In [25]:
from fastavro.schema import load_schema
import fastavro
from fastavro import parse_schema, writer
import pandas as pd
import uuid
from google.colab import files
from datetime import datetime, timedelta, timezone
import random
from faker import Faker
from timezonefinder import TimezoneFinder
import pytz
import numpy as np
from dataclasses import dataclass
from typing import Tuple

## User Personality and User Data Definition

In [53]:
from dataclasses import dataclass
from typing import Tuple

@dataclass
class ListeningPersonality:
    name: str
    genre_distribution: dict[str, float]
    active_hours: Tuple[int, int]
    interactions: dict

# Defining 8 different listening personalities
personalities = [
    ListeningPersonality(
        name="RapRockJazzLover",
        genre_distribution={"hip-hop": 0.6, "rock": 0.3, "jazz": 0.1},
        active_hours=(17, 23),  # Active from 5 PM to 11 PM
        interactions={"skip_probability": 0.1, "like_probability": 0.3, "playlist_add_probability": 0.2}
    ),
    ListeningPersonality(
        name="IndieSoulFan",
        genre_distribution={"indie": 0.5, "soul": 0.5},
        active_hours=(9, 17),  # Active from 9 AM to 5 PM
        interactions={"skip_probability": 0.15, "like_probability": 0.25, "playlist_add_probability": 0.1}
    ),
    ListeningPersonality(
        name="ElectronicExplorer",
        genre_distribution={"electronic": 0.7, "house": 0.2, "techno": 0.1},
        active_hours=(22, 4),  # Active late night to early morning
        interactions={"skip_probability": 0.05, "like_probability": 0.2, "playlist_add_probability": 0.25}
    ),
    ListeningPersonality(
        name="PopPunkPerson",
        genre_distribution={"pop": 0.4, "punk": 0.6},
        active_hours=(15, 22),
        interactions={"skip_probability": 0.2, "like_probability": 0.5, "playlist_add_probability": 0.3}
    ),
    ListeningPersonality(
        name="ClassicalConnoisseur",
        genre_distribution={"classical": 1.0},
        active_hours=(8, 20),
        interactions={"skip_probability": 0.05, "like_probability": 0.4, "playlist_add_probability": 0.15}
    ),
    ListeningPersonality(
        name="JazzJunkie",
        genre_distribution={"jazz": 0.9, "blues": 0.1},
        active_hours=(18, 24),
        interactions={"skip_probability": 0.1, "like_probability": 0.35, "playlist_add_probability": 0.25}
    ),
    ListeningPersonality(
        name="CountryCruiser",
        genre_distribution={"country": 0.8, "folk": 0.2},
        active_hours=(10, 18),
        interactions={"skip_probability": 0.12, "like_probability": 0.3, "playlist_add_probability": 0.18}
    ),
    ListeningPersonality(
        name="ReggaeRelaxer",
        genre_distribution={"reggae": 0.7, "dancehall": 0.3},
        active_hours=(16, 23),
        interactions={"skip_probability": 0.08, "like_probability": 0.4, "playlist_add_probability": 0.22}
    )
]

# Displaying the defined personalities for review
personalities

[ListeningPersonality(name='RapRockJazzLover', genre_distribution={'hip-hop': 0.6, 'rock': 0.3, 'jazz': 0.1}, active_hours=(17, 23), interactions={'skip_probability': 0.1, 'like_probability': 0.3, 'playlist_add_probability': 0.2}),
 ListeningPersonality(name='IndieSoulFan', genre_distribution={'indie': 0.5, 'soul': 0.5}, active_hours=(9, 17), interactions={'skip_probability': 0.15, 'like_probability': 0.25, 'playlist_add_probability': 0.1}),
 ListeningPersonality(name='ElectronicExplorer', genre_distribution={'electronic': 0.7, 'house': 0.2, 'techno': 0.1}, active_hours=(22, 4), interactions={'skip_probability': 0.05, 'like_probability': 0.2, 'playlist_add_probability': 0.25}),
 ListeningPersonality(name='PopPunkPerson', genre_distribution={'pop': 0.4, 'punk': 0.6}, active_hours=(15, 22), interactions={'skip_probability': 0.2, 'like_probability': 0.5, 'playlist_add_probability': 0.3}),
 ListeningPersonality(name='ClassicalConnoisseur', genre_distribution={'classical': 1.0}, active_hour

In [4]:
fake = Faker()

In [117]:
user_data = pd.read_excel('user_data.xlsx')

music_json = pd.read_json(json_file_path, orient='records', lines=True)
music_dict = music_json.to_dict('records')

In [98]:
gender_counts = user_data['Gender'].value_counts(normalize=True)
gender_options = gender_counts.index.tolist()
gender_probabilities = gender_counts.values.tolist()

age_counts = user_data['Age'].value_counts(normalize=True)
age_options = age_counts.index.tolist()
age_probabilities = age_counts.values.tolist()

subscription_plan_counts = user_data['spotify_subscription_plan'].value_counts(normalize=True)
subscription_plan_options = subscription_plan_counts.index.tolist()
subscription_plan_probabilities = subscription_plan_counts.values.tolist()

listening_device_counts = user_data['spotify_listening_device'].value_counts(normalize=True)
listening_device_options = listening_device_counts.index.tolist()
listening_device_probabilities = listening_device_counts.values.tolist()

## Schema Definition

In [ ]:
schema_ind = {
  "type": "record",
  "name": "SpotifyWrappedData",
  "namespace": "com.spotify.wrapped",
  "fields": [
    {"name": "UserId", "type": "string"},
    {"name": "Age", "type": "string"},
    {"name": "Gender", "type": "string"},
    {"name": "ListeningDevice", "type": "string"},
    {"name": "SubscriptionPlan", "type": "string"},
    {"name": "MusicTimeSlot", "type": "string"},
    {"name": "Location", "type": "string"},
    {"name": "SongId", "type": "string"},
    {"name": "Title", "type": "string"},
    {"name": "Artist", "type": "string"},
    {"name": "Genre", "type": "string"},
    {"name": "Length", "type": "string"},
    {
      "name": "InteractionType",
      "type": {
        "type": "enum",
        "name": "InteractionTypeEnum",
        "symbols": ["PLAY", "PAUSE", "SKIP", "LIKE", "ADDED_TO_PLAYLIST"]
      }
    },
    {"name": "InteractionTimestamp", "type": "long", "logicalType": "timestamp-millis"}
  ]
}

Generating for single users

In [47]:
parsed_schema = fastavro.parse_schema(schema_ind)

## Data Generation Functions

In [119]:
def generate_listening_session(user_id, personality, start_time, end_time, music_data):
    records = []
    user_country = fake.country()
    user_device = np.random.choice(listening_device_options, p=listening_device_probabilities)

    session_start = datetime.strptime(start_time, '%Y-%m-%dT%H:%M:%S%z')
    session_end = datetime.strptime(end_time, '%Y-%m-%dT%H:%M:%S%z')

    while session_start < session_end:
        genre = random.choice(list(personality.genre_distribution.keys()))

        # Filter songs by the selected genre
        filtered_songs = [song for song in music_data if song['track_genre'] == genre]

        if filtered_songs:
            song = random.choice(filtered_songs)
        else:
            print(f"No songs found for genre {genre}, selecting a random song instead.")
            song = random.choice(music_data)

        # Decide interaction type based on personality
        interaction_probabilities = [personality.interactions['skip_probability'],
                                     personality.interactions['like_probability'],
                                     personality.interactions.get('playlist_add_probability', 0)]
        interaction_types = ['SKIP', 'LIKE', 'ADDED_TO_PLAYLIST']
        interaction_type = random.choices(interaction_types + ['PLAY'], weights=interaction_probabilities + [1-sum(interaction_probabilities)], k=1)[0]


        # Generate the user profile for each record
        user_profile = {
            "UserId": user_id,
            "Age": str(np.random.choice(age_options, p=age_probabilities)),  # Ensure age is a string if required by your schema
            "Gender": np.random.choice(gender_options, p=gender_probabilities),
            "ListeningDevice": user_device,
            "SubscriptionPlan": np.random.choice(subscription_plan_options, p=subscription_plan_probabilities),
            "MusicTimeSlot": "Morning" if 5 <= session_start.hour < 12 else "Afternoon" if 12 <= session_start.hour < 18 else "Evening",
            "Location": user_country
        }

        # Structure the record to include UserProfile, SongMetadata, and UserSongInteraction
        record = {
            "UserProfile": user_profile,  # Include the generated user profile here
            "SongMetadata": {
                "SongId": song['track_id'],
                "Title": song['track_name'],
                "Artist": song['artists'],
                "Genre": song['track_genre'],
                "Length": song['duration_ms']
            },
            "UserSongInteraction": {
                "UserId": user_id,
                "SongId": str(uuid.uuid4()),  # Note: This should probably match the SongId in SongMetadata if you're tracking interactions per song
                "InteractionType": interaction_type,
                "InteractionTimestamp": session_start.strftime('%Y-%m-%dT%H:%M:%S%z')
            }
        }
        records.append(record)

        # Increment session time
        session_start += timedelta(minutes=random.randint(3, 5))  # Adjust as needed

    return records

In [118]:
def generate_multiple_users_sessions(num_users, personalities, start_time, end_time, music_data):
    all_records = []

    for _ in range(num_users):
        user_id = str(uuid.uuid4())
        # Randomly assign a listening personality to each user
        personality = random.choice(personalities)

        # Generate a listening session for each user based on their personality
        user_records = generate_listening_session(user_id, personality, start_time, end_time, music_data)
        all_records.extend(user_records)  # Add the user's records to the overall list
    print('Records Generated')

    return all_records

## Single User Generation

In [115]:
# Generating sessions for a single user .... DATA SIMULATION: 30 MINUTES
single_user_record = generate_listening_session(
    user_id=str(uuid.uuid4()),
    personality=personalities[0],
    start_time= '2024-04-01T10:00:00-05:00',
    end_time= '2024-04-01T10:30:00-05:00',
    music_data=music_dict)

In [ ]:
single_user_record

In [ ]:
# Serialization to AVRO file
output_file = 'multiple_users_spotify_data.avro'
with open(output_file, 'wb') as out:
     fastavro.writer(out, parsed_schema, single_user_record)

print(f"Generated {len(single_user_record)} records for {num_users} users and serialized to {output_file}.")

## Multiple User Generation

In [109]:
# Generate sessions for x users between specific start and end times
all_sessions = generate_multiple_users_sessions(
    num_users=3,
    personalities=personalities,
    start_time='2024-04-01T10:00:00-05:00',
    end_time='2024-04-01T12:00:00-05:00',
    music_data=music_dict
)

Records Generated


In [ ]:
all_sessions

In [ ]:
# Serialization to AVRO file
output_file = 'multiple_users_spotify_data.avro'
with open(output_file, 'wb') as out:
    fastavro.writer(out, parsed_schema, multiple_users_records)

print(f"Generated {len(multiple_users_records)} records for {num_users} users and serialized to {output_file}.")

## Saving Files

In [ ]:
files.download(output_file)